
# Triadic Cell Notebook v42
## Local real-model harness

This notebook bridges the synthetic controller line into a real local model loop.

It:

- loads a local or Hugging Face causal LM
- evaluates **multiple-choice** prompts
- uses model **conditional logprob** as the base score
- uses embeddings for:
  - prompt state
  - candidate state
  - prompt+candidate pair state
- prefers `sentence-transformers` when installed
- falls back to plain `transformers` mean-pooling if it is not installed
- runs a **v41-style Interface–Surface–Completion controller**
- compares:
  - first-pass model pick
  - controller pick

This is an **inference-time controller**, not training.

### JSONL format

```json
{"id":"q1","prompt":"Question text","choices":["a","b","c","d"],"answer_idx":2}
```

If `DATA_JSONL` is empty, the notebook uses a tiny built-in smoke-test dataset.


In [1]:

# Optional installs if needed:
# %pip install -q torch transformers sentence-transformers pandas matplotlib tqdm

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

try:
    from sentence_transformers import SentenceTransformer
    HAVE_SENTENCE_TRANSFORMERS = True
except Exception:
    SentenceTransformer = None
    HAVE_SENTENCE_TRANSFORMERS = False

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(suppress=True, precision=4)

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

print("Environment ready.")
print("sentence-transformers available:", HAVE_SENTENCE_TRANSFORMERS)


Environment ready.
sentence-transformers available: True


In [2]:

DATA_JSONL = ""  # put local jsonl path here; leave empty for built-in smoke-test dataset
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # or local path
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # or local path

MAX_SAMPLES = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

OBS_DIM = 8
CYCLES = 4
EXHALE_STEPS = 2
STEP_SIZE = 0.40
DAMPING = 0.85
ANTI_SCALE = 0.35
TOP_K_BRANCHES = 4

USE_CHAT_TEMPLATE = True
VERBOSE_EXAMPLES = 3


In [3]:

SMOKE_TEST_DATA = [
    {"id":"smoke_1","prompt":"Which planet is known as the Red Planet?","choices":["Earth","Mars","Venus","Jupiter"],"answer_idx":1},
    {"id":"smoke_2","prompt":"What gas do plants primarily absorb from the atmosphere?","choices":["Oxygen","Hydrogen","Carbon dioxide","Nitrogen"],"answer_idx":2},
    {"id":"smoke_3","prompt":"What is the capital of Japan?","choices":["Beijing","Seoul","Tokyo","Kyoto"],"answer_idx":2},
    {"id":"smoke_4","prompt":"Which number is prime?","choices":["21","35","37","49"],"answer_idx":2},
]

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_rows() -> List[Dict[str, Any]]:
    rows = load_jsonl(DATA_JSONL) if DATA_JSONL and Path(DATA_JSONL).exists() else SMOKE_TEST_DATA.copy()
    cleaned = []
    for i, row in enumerate(rows[:MAX_SAMPLES]):
        if not all(k in row for k in ("prompt","choices","answer_idx")):
            continue
        if not isinstance(row["choices"], list) or len(row["choices"]) < 2:
            continue
        cleaned.append({
            "id": row.get("id", f"row_{i}"),
            "prompt": str(row["prompt"]),
            "choices": [str(x) for x in row["choices"]],
            "answer_idx": int(row["answer_idx"]),
        })
    return cleaned

rows = load_rows()
len(rows), rows[:2]


(4,
 [{'id': 'smoke_1',
   'prompt': 'Which planet is known as the Red Planet?',
   'choices': ['Earth', 'Mars', 'Venus', 'Jupiter'],
   'answer_idx': 1},
  {'id': 'smoke_2',
   'prompt': 'What gas do plants primarily absorb from the atmosphere?',
   'choices': ['Oxygen', 'Hydrogen', 'Carbon dioxide', 'Nitrogen'],
   'answer_idx': 2}])

In [ ]:

def load_generation_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    model.eval()
    return tokenizer, model

class TransformerEmbedder:
    def __init__(self, model_name: str, device: str):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=DTYPE,
        )
        if DEVICE != "cuda":
            self.model = self.model.to(device)
        self.model.eval()

    def encode(self, texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True):
        all_vecs = []
        iterator = range(0, len(texts), batch_size)
        if show_progress_bar:
            iterator = tqdm(iterator, total=(len(texts) + batch_size - 1) // batch_size, desc="Embedding")
        for start in iterator:
            batch = texts[start:start + batch_size]
            toks = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            )
            toks = {k: v.to(self.device) for k, v in toks.items()}
            with torch.no_grad():
                out = self.model(**toks)
                hidden = out.last_hidden_state
                attn = toks["attention_mask"].unsqueeze(-1)
                summed = (hidden * attn).sum(dim=1)
                denom = attn.sum(dim=1).clamp_min(1)
                vec = summed / denom
                if normalize_embeddings:
                    vec = F.normalize(vec, p=2, dim=1)
            all_vecs.append(vec.detach().cpu().numpy())

        arr = np.vstack(all_vecs).astype(np.float32)
        return arr if convert_to_numpy else arr

def load_embedder(model_name: str):
    if HAVE_SENTENCE_TRANSFORMERS:
        return SentenceTransformer(model_name, device=DEVICE)
    return TransformerEmbedder(model_name, device=DEVICE)

tokenizer, lm = load_generation_model(MODEL_NAME)
embedder = load_embedder(EMBED_MODEL_NAME)

print("Loaded generation model:", MODEL_NAME)
print("Loaded embed model:", EMBED_MODEL_NAME)
print("Embed backend:", "sentence-transformers" if HAVE_SENTENCE_TRANSFORMERS else "transformers-mean-pool")
print("Device:", DEVICE)


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

In [ ]:

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def build_mcq_prompt(row: Dict[str, Any]) -> str:
    parts = [row["prompt"].strip(), "", "Choices:"]
    for i, choice in enumerate(row["choices"]):
        parts.append(f"{LETTERS[i]}. {choice}")
    parts.append("")
    parts.append("Answer with the single correct letter only.")
    return "\n".join(parts)

def maybe_apply_chat_template(prompt: str) -> str:
    if not USE_CHAT_TEMPLATE or not hasattr(tokenizer, "apply_chat_template"):
        return prompt
    try:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return prompt

def conditional_logprob(prefix: str, suffix: str) -> float:
    prefix_ids = tokenizer(prefix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    full_ids = tokenizer(prefix + suffix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)

    with torch.no_grad():
        outputs = lm(full_ids)
        logits = outputs.logits[:, :-1, :]
        targets = full_ids[:, 1:]
        logprobs = F.log_softmax(logits, dim=-1)
        token_lp = logprobs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)

    start = prefix_ids.shape[1] - 1
    suffix_lp = token_lp[:, start:]
    return float(suffix_lp.mean().item())

def score_choice_letters(row: Dict[str, Any]) -> Dict[str, Any]:
    base_prompt = build_mcq_prompt(row)
    rendered_prompt = maybe_apply_chat_template(base_prompt)
    scores = []
    for i, _ in enumerate(row["choices"]):
        lp = conditional_logprob(rendered_prompt, " " + LETTERS[i])
        scores.append(lp)
    pred_idx = int(np.argmax(scores))
    return {
        "prompt_text": base_prompt,
        "rendered_prompt": rendered_prompt,
        "base_scores": np.array(scores, dtype=np.float32),
        "base_pred_idx": pred_idx,
    }


In [ ]:

def l2norm(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.clip(n, eps, None)

def minmax_local(stack: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    mn = stack.min(axis=0, keepdims=True)
    mx = stack.max(axis=0, keepdims=True)
    return (stack - mn) / np.clip(mx - mn, eps, None)

def embed_texts(texts: List[str], batch_size: int = 64) -> np.ndarray:
    emb = embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return emb.astype(np.float32)

@dataclass
class PreparedSample:
    row_id: str
    prompt_text: str
    choices: List[str]
    answer_idx: int
    base_scores: np.ndarray
    base_pred_idx: int
    prompt_vec: np.ndarray
    candidate_vecs: np.ndarray
    pair_vecs: np.ndarray

def prepare_samples(rows: List[Dict[str, Any]]) -> List[PreparedSample]:
    scored = []
    all_prompt_texts = []
    all_choice_texts = []
    all_pair_texts = []

    for row in tqdm(rows, desc="Base model scoring"):
        s = score_choice_letters(row)
        scored.append((row, s))
        all_prompt_texts.append(s["prompt_text"])
        for i, choice in enumerate(row["choices"]):
            all_choice_texts.append(f"{LETTERS[i]}. {choice}")
            all_pair_texts.append(s["prompt_text"] + "\n\nCandidate: " + f"{LETTERS[i]}. {choice}")

    prompt_embs = embed_texts(all_prompt_texts)
    choice_embs = embed_texts(all_choice_texts)
    pair_embs = embed_texts(all_pair_texts)

    prepared = []
    choice_ptr = 0
    pair_ptr = 0

    for prompt_idx, (row, s) in enumerate(scored):
        n = len(row["choices"])
        cand = choice_embs[choice_ptr:choice_ptr+n]
        pair = pair_embs[pair_ptr:pair_ptr+n]

        stack = np.vstack([prompt_embs[prompt_idx:prompt_idx+1], cand, pair]).astype(np.float32)
        stack = l2norm(stack)
        stack = minmax_local(stack)

        prepared.append(
            PreparedSample(
                row_id=row["id"],
                prompt_text=s["prompt_text"],
                choices=row["choices"],
                answer_idx=row["answer_idx"],
                base_scores=s["base_scores"],
                base_pred_idx=s["base_pred_idx"],
                prompt_vec=stack[0],
                candidate_vecs=stack[1:1+n],
                pair_vecs=stack[1+n:1+n+n],
            )
        )

        choice_ptr += n
        pair_ptr += n

    return prepared

prepared_samples = prepare_samples(rows)
len(prepared_samples)


In [ ]:

def orthonormal_obs_w(dim: int, obs_dim: int, seed: int = 42) -> np.ndarray:
    local_rng = np.random.default_rng(seed)
    M = local_rng.normal(size=(obs_dim, dim)).astype(np.float32)
    Q = []
    for row in M:
        u = row.copy()
        for q in Q:
            u = u - np.dot(u, q) * q
        n = np.linalg.norm(u)
        if n > 1e-8:
            Q.append(u / n)
    Q = np.stack(Q, axis=0)
    if Q.shape[0] < obs_dim:
        Q = np.vstack([Q, np.zeros((obs_dim - Q.shape[0], dim), dtype=np.float32)])
    return Q[:obs_dim]

OBS_W = orthonormal_obs_w(prepared_samples[0].prompt_vec.shape[0], OBS_DIM, seed=SEED)

def cosine_similarity_np(a: np.ndarray, b: np.ndarray) -> float:
    na = np.linalg.norm(a) + 1e-8
    nb = np.linalg.norm(b) + 1e-8
    return float(np.dot(a, b) / (na * nb))

def observable_from_latent(vec: np.ndarray) -> np.ndarray:
    return np.clip(OBS_W @ vec, 0.0, 1.0)

def apply_resoluteness(x: np.ndarray, rho: float, eps: float = 1e-8) -> np.ndarray:
    return np.clip(np.clip(x, eps, 1.0) ** rho, 0.0, 1.0)

def build_gap_masks(pred_obs: np.ndarray) -> np.ndarray:
    k = pred_obs.shape[0]
    out = np.zeros_like(pred_obs)
    for i in range(k):
        others = [pred_obs[j] for j in range(k) if j != i]
        other_mean = np.mean(np.stack(others, axis=0), axis=0) if others else np.zeros_like(pred_obs[i])
        out[i] = np.clip(np.abs(pred_obs[i] - other_mean), 0.0, 1.0)
    return out

def softmax_np(x: np.ndarray) -> np.ndarray:
    z = x - np.max(x)
    e = np.exp(z)
    return e / (e.sum() + 1e-8)

def branch_entropy(scores: np.ndarray) -> float:
    p = softmax_np(scores)
    return float(-np.sum(p * np.log(p + 1e-8)))

def build_request_shape(shared_matter, candidate_vec, gap_mask, rho=1.0, top_k=1):
    pred_obs = observable_from_latent(candidate_vec)
    vacuum = np.clip(0.75 * (1.0 - shared_matter) + 0.25 * np.maximum(shared_matter - candidate_vec, 0.0), 0.0, 1.0)
    vacuum_obs = observable_from_latent(vacuum)

    vacuum_obs_r = apply_resoluteness(vacuum_obs, rho)
    need_obs_r = apply_resoluteness(1.0 - pred_obs, rho)
    gap_obs_r = apply_resoluteness(gap_mask, rho)

    scores = vacuum_obs_r * need_obs_r * (0.5 + 0.5 * gap_obs_r)
    idx = np.argsort(scores)[-top_k:]

    mask = np.zeros_like(scores)
    mask[idx] = 1.0

    t = min(max((rho - 0.35) / 1.10, 0.0), 1.0)
    tol_lo, tol_hi = 0.08, 0.30
    tolerance = tol_hi * (1.0 - t) + tol_lo * t

    return {
        "pred_obs": pred_obs,
        "vacuum_obs": vacuum_obs_r,
        "gap_obs": gap_obs_r,
        "mask": mask,
        "tolerance": float(tolerance),
    }

def masked_error(obs_packet, pred_obs, mask):
    return float(np.mean(np.abs(mask * (obs_packet - pred_obs))))

def rho_from_field(mean_seat_gain, mean_obs_misfit, winner_margin, entropy):
    raw = 4.0 * mean_seat_gain - 3.0 * mean_obs_misfit + 2.0 * winner_margin - 1.5 * entropy
    sig = 1.0 / (1.0 + np.exp(-raw))
    return float(0.35 + 1.10 * sig)


In [ ]:

def controller_v42(sample: PreparedSample, cycles=CYCLES, exhale_steps=EXHALE_STEPS, step_size=STEP_SIZE, damping=DAMPING, anti_scale=ANTI_SCALE):
    k = len(sample.choices)
    matter = sample.prompt_vec.copy()
    candidate_vecs = sample.candidate_vecs.copy()
    pair_vecs = sample.pair_vecs.copy()

    base_scores = sample.base_scores.astype(np.float32)
    base_z = (base_scores - base_scores.mean()) / (base_scores.std() + 1e-8)

    rho = 1.0
    prev_completion = np.zeros(k, dtype=np.float32)
    trace_rows = []
    last_ranked = None

    for cycle in range(cycles):
        seat_gains_all = []
        misfits_all = []

        for _ in range(exhale_steps):
            pred_obs = np.stack([observable_from_latent(candidate_vecs[i]) for i in range(k)], axis=0)
            pair_obs = np.stack([observable_from_latent(pair_vecs[i]) for i in range(k)], axis=0)
            gap_masks = build_gap_masks(pred_obs)

            requests = [build_request_shape(matter, candidate_vecs[i], gap_masks[i], rho=rho, top_k=1) for i in range(k)]

            completion_memory = np.zeros(k, dtype=np.float32)
            fluxes = np.zeros_like(candidate_vecs)

            matter_before = matter.copy()

            for i in range(k):
                req = requests[i]
                mask = req["mask"]
                self_fit = 1.0 - masked_error(pair_obs[i], req["pred_obs"], mask)
                rival_fits = [1.0 - masked_error(pair_obs[i], pred_obs[j], mask) for j in range(k) if j != i]
                rival_fit_mean = float(np.mean(rival_fits)) if rival_fits else 0.0

                completion_now = float(self_fit - anti_scale * rival_fit_mean + 0.20 * base_z[i])
                completion_memory[i] = 0.65 * completion_now + 0.35 * prev_completion[i]

                missing = np.maximum(candidate_vecs[i] - matter, 0.0)
                excess = np.maximum(matter - candidate_vecs[i], 0.0)
                signed_correction = missing - 0.6 * excess

                pair_back = pair_obs[i] @ OBS_W
                gap_back = req["gap_obs"] @ OBS_W
                interface_vacuum = np.clip(0.75 * (1.0 - matter) + 0.25 * excess, 0.0, 1.0)

                others = [pred_obs[j] for j in range(k) if j != i]
                other_mean = np.mean(np.stack(others, axis=0), axis=0) if others else np.zeros_like(pred_obs[i])

                completion_obs = mask * (pair_obs[i] - pred_obs[i]) - anti_scale * mask * (other_mean - pair_obs[i])
                completion_back = completion_obs @ OBS_W

                anti_obs = mask * (other_mean - pred_obs[i])
                anti_back = anti_obs @ OBS_W

                fluxes[i] = (
                    0.55 * signed_correction
                    + 0.20 * (pair_back - 0.5)
                    + 0.20 * (apply_resoluteness(interface_vacuum, rho) - 0.5)
                    + 0.15 * (gap_back - 0.5)
                    + 0.45 * completion_back
                    - 0.35 * anti_back
                )

                misfits_all.append(masked_error(pair_obs[i], pred_obs[i], mask))

            weights = softmax_np(completion_memory + 0.25 * base_z)
            total_flux = np.sum(weights[:, None] * fluxes, axis=0)

            matter_after = np.clip(damping * matter + (1.0 - damping) * np.clip(matter + step_size * total_flux, 0.0, 1.0), 0.0, 1.0)

            for i in range(k):
                before_missing = float(np.maximum(candidate_vecs[i] - matter_before, 0.0).mean())
                after_missing = float(np.maximum(candidate_vecs[i] - matter_after, 0.0).mean())
                before_excess = float(np.maximum(matter_before - candidate_vecs[i], 0.0).mean())
                after_excess = float(np.maximum(matter_after - candidate_vecs[i], 0.0).mean())
                before_fit = cosine_similarity_np(matter_before, candidate_vecs[i])
                after_fit = cosine_similarity_np(matter_after, candidate_vecs[i])
                sg = 0.60 * (before_missing - after_missing) - 0.25 * (after_excess - before_excess) + 0.15 * (after_fit - before_fit)
                seat_gains_all.append(float(sg))

            matter = matter_after
            prev_completion = completion_memory

        pred_obs = np.stack([observable_from_latent(candidate_vecs[i]) for i in range(k)], axis=0)
        pair_obs = np.stack([observable_from_latent(pair_vecs[i]) for i in range(k)], axis=0)
        gap_masks = build_gap_masks(pred_obs)

        cycle_items = []
        for i in range(k):
            req = build_request_shape(matter, candidate_vecs[i], gap_masks[i], rho=rho, top_k=1)
            mask = req["mask"]
            self_err = masked_error(pair_obs[i], req["pred_obs"], mask)
            rival_errs = [masked_error(pair_obs[i], pred_obs[j], mask) for j in range(k) if j != i]
            rival_err = float(np.mean(rival_errs)) if rival_errs else self_err

            advantage = float(rival_err - self_err)
            completion_score = float(0.65 * advantage + 0.35 * prev_completion[i] + 0.15 * base_z[i])

            missing_mass = float(np.maximum(candidate_vecs[i] - matter, 0.0).mean())
            retrieve_mass = float(np.minimum(candidate_vecs[i], matter).mean())
            contradiction_mass = float(np.maximum(matter - candidate_vecs[i], 0.0).mean())
            declaration_score = float(cosine_similarity_np(matter, candidate_vecs[i]) + 0.35 * base_z[i])

            energy = (
                0.76 * missing_mass
                + 0.55 * contradiction_mass
                - 0.78 * declaration_score
                - 0.30 * retrieve_mass
                + 0.45 * self_err
                - 0.65 * completion_score
            )

            cycle_items.append({
                "idx": i,
                "energy": float(energy),
                "advantage": float(advantage),
                "completion_score": float(completion_score),
                "self_err": float(self_err),
                "rival_err": float(rival_err),
                "missing_mass": missing_mass,
                "retrieve_mass": retrieve_mass,
                "contradiction_mass": contradiction_mass,
                "declaration_score": declaration_score,
                "mask": req["mask"],
                "gap_obs": req["gap_obs"],
                "vacuum_obs": req["vacuum_obs"],
                "tolerance": req["tolerance"],
            })

        ranked = sorted(cycle_items, key=lambda x: x["energy"])
        winner_margin = float(ranked[1]["energy"] - ranked[0]["energy"]) if len(ranked) > 1 else 1.0
        entropy = branch_entropy(np.array([-x["energy"] for x in ranked], dtype=np.float32))
        rho = rho_from_field(
            float(np.mean(seat_gains_all)) if seat_gains_all else 0.0,
            float(np.mean(misfits_all)) if misfits_all else 0.0,
            winner_margin,
            entropy,
        )

        for item in cycle_items:
            trace_rows.append({
                "cycle": cycle,
                "choice_idx": item["idx"],
                "choice": sample.choices[item["idx"]],
                "rho": float(rho),
                "energy": item["energy"],
                "advantage": item["advantage"],
                "completion_score": item["completion_score"],
                "self_err": item["self_err"],
                "rival_err": item["rival_err"],
                "missing_mass": item["missing_mass"],
                "retrieve_mass": item["retrieve_mass"],
                "contradiction_mass": item["contradiction_mass"],
                "declaration_score": item["declaration_score"],
                "request_mask": item["mask"].tolist(),
                "gap_obs": item["gap_obs"].tolist(),
                "vacuum_obs": item["vacuum_obs"].tolist(),
                "tolerance": item["tolerance"],
            })

        last_ranked = ranked

    pred_idx = int(last_ranked[0]["idx"])
    return {
        "pred_idx": pred_idx,
        "trace_df": pd.DataFrame(trace_rows),
        "final_matter": matter,
    }


In [ ]:

results = []
trace_cache = {}

for sample in tqdm(prepared_samples, desc="Controller evaluation"):
    out = controller_v42(sample)
    trace_cache[sample.row_id] = out["trace_df"]

    results.append({
        "id": sample.row_id,
        "prompt": sample.prompt_text,
        "answer_idx": sample.answer_idx,
        "base_pred_idx": sample.base_pred_idx,
        "ctrl_pred_idx": out["pred_idx"],
        "base_correct": int(sample.base_pred_idx == sample.answer_idx),
        "ctrl_correct": int(out["pred_idx"] == sample.answer_idx),
        "base_choice": sample.choices[sample.base_pred_idx],
        "ctrl_choice": sample.choices[out["pred_idx"]],
        "gold_choice": sample.choices[sample.answer_idx],
    })

results_df = pd.DataFrame(results)
results_df.head()


In [ ]:

compare_summary = pd.DataFrame([{
    "n_samples": int(len(results_df)),
    "base_accuracy": float(results_df["base_correct"].mean()),
    "controller_accuracy": float(results_df["ctrl_correct"].mean()),
    "gain": float(results_df["ctrl_correct"].mean() - results_df["base_correct"].mean()),
}])
compare_summary


In [ ]:

mode_rows = []
for sample in prepared_samples:
    scores = sample.base_scores
    order = np.sort(scores)[::-1]
    margin = float(order[0] - order[1]) if len(order) > 1 else 0.0
    difficulty = "ambiguous" if margin < 0.15 else ("partial_like" if margin < 0.5 else "easy")
    res = results_df[results_df["id"] == sample.row_id].iloc[0]
    mode_rows.append({
        "difficulty": difficulty,
        "base_correct": int(res["base_correct"]),
        "ctrl_correct": int(res["ctrl_correct"]),
        "improved": int((not res["base_correct"]) and res["ctrl_correct"]),
    })

by_mode_compare = (
    pd.DataFrame(mode_rows)
    .groupby("difficulty")[["base_correct", "ctrl_correct", "improved"]]
    .mean()
    .reset_index()
)
by_mode_compare


In [ ]:

for row_id in list(trace_cache.keys())[:VERBOSE_EXAMPLES]:
    print("=" * 80)
    print("ID:", row_id)
    display(results_df[results_df["id"] == row_id][["prompt", "gold_choice", "base_choice", "ctrl_choice"]])
    display(trace_cache[row_id].head(12))


In [ ]:

plt.figure(figsize=(8, 4))
compare_summary[["base_accuracy", "controller_accuracy"]].T.plot(kind="bar", legend=False)
plt.title("Base vs Controller Accuracy")
plt.ylabel("accuracy")
plt.xticks(rotation=0)
plt.show()

if len(by_mode_compare):
    plt.figure(figsize=(8, 4))
    by_mode_compare.set_index("difficulty")[["base_correct", "ctrl_correct"]].plot(kind="bar")
    plt.title("Accuracy by difficulty bucket")
    plt.ylabel("accuracy")
    plt.xticks(rotation=0)
    plt.show()

if len(trace_cache):
    first_key = list(trace_cache.keys())[0]
    trace_df = trace_cache[first_key]

    plt.figure(figsize=(10, 4))
    for choice_idx in sorted(trace_df["choice_idx"].unique()):
        df = trace_df[trace_df["choice_idx"] == choice_idx]
        plt.plot(df["cycle"], df["energy"], marker="o", label=f"choice {choice_idx}")
    plt.title(f"Energy by Cycle: {first_key}")
    plt.xlabel("cycle")
    plt.ylabel("energy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 4))
    trace_df.groupby("cycle")["completion_score"].mean().plot(marker="o")
    plt.title(f"Mean Completion Score by Cycle: {first_key}")
    plt.xlabel("cycle")
    plt.ylabel("completion_score")
    plt.show()



## What to change locally

- point `DATA_JSONL` at your dataset
- point `MODEL_NAME` at your local LM
- reduce `MAX_SAMPLES` first
- if the controller over-steers, reduce:
  - `ANTI_SCALE`
  - `STEP_SIZE`
- if it is too weak, raise:
  - `CYCLES`
  - `EXHALE_STEPS`
  - `ANTI_SCALE`

Read first:
- `compare_summary`
- `by_mode_compare`
- `results_df`
- per-example `trace_df`
